In [ ]:
import anthropic
import json
import time
import random
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=api_key)

OUTPUT_PATH = "C:/datasets/llm-data/sft"
TARGET_SAMPLES = 100

In [ ]:
DS_TOPICS = [
    "linear regression", "overfitting", "confusion matrix",
    "ROC curve", "k means clustering", "PCA",
    "correlation heatmap", "histogram"
]

STATS_TOPICS = [
    "mean and variance", "normal distribution",
    "central limit theorem", "hypothesis testing",
    "confidence intervals", "bayes theorem"
]

MATH_TOPICS = [
    "derivative of x^2", "integration basics",
    "matrix multiplication", "gradient of a function",
    "logarithmic functions"
]

ALL_TOPICS = DS_TOPICS + STATS_TOPICS + MATH_TOPICS

In [ ]:
def get_prompt(topic):
    return f"""
Generate ONE high-quality analytical instruction example.

Topic: {topic}

Return ONLY valid JSON:

{{
  "instruction": "...",
  "input": "",
  "output": "### Reasoning:\\nStep 1: ...\\nStep 2: ...\\n\\n### Code:\\n<python code without backticks>\\n\\n### Answer:\\n..."
}}

Rules:
- Step-by-step reasoning
- Code must run (numpy, matplotlib, pandas, seaborn, or sympy)
- No markdown ```
- Keep answer concise
"""


In [ ]:
def generate_sample(topic):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=700,  # keep low to save money
        messages=[{"role": "user", "content": get_prompt(topic)}]
    )

    text = response.content[0].text.strip()
    return json.loads(text)

In [ ]:
def validate(sample):
    if not all(k in sample for k in ["instruction", "input", "output"]):
        return False
    
    out = sample["output"]
    required = ["### Reasoning:", "### Code:", "### Answer:"]
    
    return all(r in out for r in required)

In [ ]:
dataset = []
attempts = 0
MAX_ATTEMPTS = 300  

print("Generating dataset...")

while len(dataset) < TARGET_SAMPLES and attempts < MAX_ATTEMPTS:
    topic = random.choice(ALL_TOPICS)
    attempts += 1
    
    try:
        sample = generate_sample(topic)
        
        if validate(sample):
            dataset.append(sample)
            print(f"✓ {len(dataset)}/{TARGET_SAMPLES} | {topic}")
        else:
            print("⚠️ Invalid format")
    
    except Exception as e:
        print("Error:", e)
    
    time.sleep(0.5)  

print(f"\nDone: {len(dataset)} samples")

In [ ]:
dataset[:3]